# Домашнє завдання 3. Coffee Quality & Pricing
**Автор:** Лариса Дяченко · **Джерело:** Kaggle, Razan Ihab, ліцензія CC0. Дані синтетичні.

## A. Огляд даних
Один рядок – одна партія кави: країна, ферма, обробка, лабораторні вимірювання, оцінка якості й ціна за кг.

In [1]:
import pandas as pd
df = pd.read_csv("coffee_value_benchmark.csv")
print(df.shape)   # (35700, 36)

(35700, 36)


In [2]:
df.head()

,record_id,lot_id,harvest_season,harvest_calendar_year,harvest_month,country_of_origin,farm_type,farm_size_hectares,tree_age_years,altitude_mean_meters,...,lot_size_kg,roast_level,acidity_score,sweetness_score,body_score,aroma_score,flavor_score,coffee_quality_score,quality_grade,price_usd_per_kg
0,REC-0000001,LOT-0032936,2020/21,2020,10,Vietnam,Smallholder,0.58,28.0,1961.0,...,26433.0,Medium,7.95,7.30,8.45,7.35,7.37,78.01,Very Good,2.43
1,REC-0000002,LOT-0030952,2021/22,2022,1,Honduras,Smallholder,1.12,2.0,1257.0,...,25446.0,Medium,7.62,8.17,7.45,7.80,6.88,75.62,Good,4.06
2,REC-0000003,LOT-0018572,2020/21,2020,11,Vietnam,Cooperative,13.82,5.0,557.0,...,15707.0,Medium,8.47,8.83,8.13,7.70,8.10,72.41,Commercial,4.26
3,REC-0000004,LOT-0005318,2021/22,2021,10,Vietnam,Cooperative,9.96,22.0,652.0,...,2879.0,Medium,7.54,8.12,8.21,7.77,7.13,75.67,Good,3.60
4,REC-0000005,LOT-0006317,2021/22,2021,7,Brazil,Smallholder,1.90,11.0,971.0,...,9561.0,Medium,6.70,8.08,8.68,7.38,7.16,73.77,Good,4.16


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35700 entries, 0 to 35699
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   record_id               35700 non-null  str    
 1   lot_id                  35700 non-null  str    
 2   harvest_season          35700 non-null  str    
 3   harvest_calendar_year   35700 non-null  int64  
 4   harvest_month           35700 non-null  int64  
 5   country_of_origin       35700 non-null  str    
 6   farm_type               35700 non-null  str    
 7   farm_size_hectares      35700 non-null  float64
 8   tree_age_years          35700 non-null  float64
 9   altitude_mean_meters    35700 non-null  float64
 10  species                 35700 non-null  str    
 11  avg_temp_c              35700 non-null  float64
 12  avg_annual_rainfall_mm  33881 non-null  float64
 13  humidity_pct            33418 non-null  float64
 14  processing_method       35700 non-null  str    
 

In [5]:
df.isna().sum()

record_id                     0
lot_id                        0
harvest_season                0
harvest_calendar_year         0
harvest_month                 0
country_of_origin             0
farm_type                     0
farm_size_hectares            0
tree_age_years                0
altitude_mean_meters          0
species                       0
avg_temp_c                    0
avg_annual_rainfall_mm     1819
humidity_pct               2282
processing_method             0
drying_method                 0
drying_duration_days          0
storage_months                0
storage_conditions            0
bean_size_screen              0
moisture_pct               1612
bean_density_g_ml          1632
defect_rate_pct               0
traceability_level            0
certification             21723
intended_buyer_segment        0
lot_size_kg                   0
roast_level                   0
acidity_score                 0
sweetness_score               0
body_score                    0
aroma_sc

In [7]:
missing = df.isna().sum()
missing[missing > 0]

avg_annual_rainfall_mm     1819
humidity_pct               2282
moisture_pct               1612
bean_density_g_ml          1632
certification             21723
dtype: int64

Пропуски є у 5 стовпцях. Найбільше – у `certification`; його розбираю нижче (ризик 3), після очищення від дублікатів.

In [119]:
df.describe()

,harvest_calendar_year,harvest_month,farm_size_hectares,tree_age_years,altitude_mean_meters,avg_temp_c,avg_annual_rainfall_mm,humidity_pct,drying_duration_days,storage_months,...,bean_density_g_ml,defect_rate_pct,lot_size_kg,acidity_score,sweetness_score,body_score,aroma_score,flavor_score,coffee_quality_score,price_usd_per_kg
count,35700.000000,35700.000000,35700.000000,35700.000000,35700.000000,35700.000000,33881.000000,33418.000000,35700.000000,35700.000000,...,34068.000000,35700.000000,35700.000000,35700.000000,35700.000000,35700.000000,35700.000000,35700.000000,35700.000000,35700.000000
mean,2021.470308,8.091401,16.588095,15.140364,1145.550896,18.182387,1884.958886,74.532743,14.277216,3.996723,...,1.121776,2.769994,18057.653501,7.742544,7.899808,8.347656,7.603614,7.518959,77.602334,4.813070
std,1.282059,3.230757,37.377621,8.013493,531.533991,4.432564,530.862432,6.099831,5.039598,2.001782,...,0.049839,2.101308,11412.570123,0.952912,0.729056,0.638657,0.754433,0.696280,4.915255,2.199144
min,2019.000000,1.000000,0.350000,2.000000,150.000000,7.100000,327.000000,55.900000,2.000000,0.000000,...,0.962000,0.000000,20.000000,6.000000,6.000000,6.000000,6.000000,6.000000,60.000000,1.780000
25%,2021.000000,6.000000,1.570000,9.000000,692.000000,14.100000,1453.000000,69.800000,11.000000,3.000000,...,1.081000,1.200000,7559.750000,7.020000,7.390000,7.910000,7.070000,7.030000,73.810000,3.460000
50%,2021.000000,9.000000,2.750000,14.000000,1071.000000,19.000000,1883.000000,74.200000,14.600000,4.000000,...,1.129000,2.500000,17523.000000,7.790000,7.920000,8.350000,7.620000,7.550000,78.240000,4.180000
75%,2022.000000,11.000000,14.990000,19.000000,1616.000000,22.000000,2303.000000,79.400000,17.800000,5.000000,...,1.160000,4.000000,27812.250000,8.460000,8.410000,8.790000,8.130000,8.020000,81.330000,5.360000
max,2024.000000,12.000000,920.260000,50.000000,2726.000000,27.000000,3345.000000,93.300000,29.200000,15.000000,...,1.269000,17.000000,39990.000000,9.800000,9.800000,9.800000,9.800000,9.800000,94.180000,22.850000


## B. Ризики якості даних

### Ризик 1: викиди в площі ферм
Аналіз площі й ціни в цьому розділі зроблено на `df`, до видалення дублікатів; 700 повторів із 35 700 (2 %) на висновки не впливають.

In [120]:
df["farm_size_hectares"].describe()

count    35700.000000
mean        16.588095
std         37.377621
min          0.350000
25%          1.570000
50%          2.750000
75%         14.990000
max        920.260000
Name: farm_size_hectares, dtype: float64

In [121]:
farm = df["farm_size_hectares"]
print(round(farm.max() / farm.quantile(0.75)))

61


75 % ферм мають площу до 15 га, а найбільша – 920 га, тобто приблизно в 61 раз більша за цю межу.

In [122]:
df.groupby("farm_type")["farm_size_hectares"].median()

farm_type
Cooperative    12.24
Estate         54.52
Smallholder     1.65
Name: farm_size_hectares, dtype: float64

У наборі змішані три дуже різні типи ферм. Типова плантація в 33 рази більша за типове дрібне господарство (54.52 ÷ 1.65). Тому загальна медіана (2.75) і середнє (16.59) так різняться: дрібних ферм найбільше, а кілька тисяч плантацій «тягнуть» середнє вгору.

In [123]:
df[df["farm_size_hectares"] > 500]["farm_type"].value_counts()

farm_type
Estate    15
Name: count, dtype: int64

**Ризик 1: викиди у `farm_size_hectares`** (max ≈ 920 га, у 61 раз більше за 75-й процентиль, 14.99 га).

**Як виявили:** `describe()`, потім `groupby("farm_type")["farm_size_hectares"].median()` і фільтр `> 500`.

**Результат:** медіана площі різна для типів ферм: Smallholder 1.65 га, Cooperative 12.24 га, Estate 54.52 га. Площу понад 500 га мають лише 15 записів, і всі вони – Estate.

**Рішення:** не видаляю, бо великі плантації реально можуть мати сотні гектарів. Типову площу описую медіаною, а площі порівнюю в межах типу ферми, щоб великі плантації не спотворювали результат.

### Чи впливає розмір ферми на ціну?

In [124]:
df["farm_size_hectares"].corr(df["price_usd_per_kg"]) 

np.float64(0.2679174530722346)

Загальна кореляція площі й ціни r ≈ 0.27 (|r| < 0.3): зв'язок слабкий, але додатний.

In [125]:
df.groupby("farm_type")["price_usd_per_kg"].median()

farm_type
Cooperative    4.45
Estate         5.51
Smallholder    3.87
Name: price_usd_per_kg, dtype: float64

Кава з великих господарств найдорожча.

In [126]:
df.groupby("farm_type")[["farm_size_hectares", "price_usd_per_kg"]].corr()

farm_size_hectares  price_usd_per_kg
farm_type                                                           
Cooperative farm_size_hectares            1.000000          0.016551
            price_usd_per_kg              0.016551          1.000000
Estate      farm_size_hectares            1.000000         -0.005386
            price_usd_per_kg             -0.005386          1.000000
Smallholder farm_size_hectares            1.000000         -0.000857
            price_usd_per_kg             -0.000857          1.000000

Усередині кожного типу ферми кореляція площі й ціни майже нульова (від −0.005 до 0.017).

Висновок: на ціну впливає тип господарства, а не площа. Площа лише «їде разом» із типом. 

In [127]:
df.groupby("farm_type")["coffee_quality_score"].median()

farm_type
Cooperative    78.26
Estate         78.22
Smallholder    78.22
Name: coffee_quality_score, dtype: float64

Якість не відрізняється залежно від типу господарства (медіана ≈ 78.2), тож вищу ціну плантацій якістю не пояснити.

In [128]:
df.groupby("traceability_level")["price_usd_per_kg"].median().sort_values()

traceability_level
Country        3.67
Region         4.01
Cooperative    4.86
Farm           5.09
Microlot       9.90
Name: price_usd_per_kg, dtype: float64

Мікролот коштує майже втричі дорожче за каву, про яку відомо лише країну (9.90 ÷ 3.67 ≈ 2.7). Покупці платять за «історію» партії.

In [129]:
pd.crosstab(df["farm_type"], df["traceability_level"], normalize="index").round(2)

traceability_level,Cooperative,Country,Farm,Microlot,Region
farm_type,,,,,
Cooperative,0.70,0.1,0.0,0.00,0.20
Estate,0.00,0.0,0.6,0.31,0.09
Smallholder,0.11,0.5,0.0,0.00,0.38


Висновок: дорожче продають не більші ферми, а ті, що можуть точніше показати походження кави. Площа лише «їде разом» із типом ферми.

### Ризик 2: дублікати

In [130]:
df.duplicated().sum()

np.int64(0)

In [131]:
df["record_id"].is_unique

True

In [132]:
df["lot_id"].duplicated().sum()

np.int64(700)

In [133]:
df.drop(columns=["record_id"]).duplicated().sum()

np.int64(350)

In [134]:
dup_lots = df[df["lot_id"].duplicated(keep=False)]
dup_lots.sort_values("lot_id").head(6)

,record_id,lot_id,harvest_season,harvest_calendar_year,harvest_month,country_of_origin,farm_type,farm_size_hectares,tree_age_years,altitude_mean_meters,...,lot_size_kg,roast_level,acidity_score,sweetness_score,body_score,aroma_score,flavor_score,coffee_quality_score,quality_grade,price_usd_per_kg
2884,REC-0002885,LOT-0000062,2022/23,2023,4,Colombia,Estate,141.17,20.0,2300.0,...,876.0,Light,9.50,8.15,8.46,7.63,8.47,84.28,Excellent,5.52
652,REC-0000653,LOT-0000062,2022/23,2023,4,Colombia,Estate,141.17,20.0,2300.0,...,876.0,Light,9.50,8.15,8.46,7.63,8.47,84.28,Excellent,5.52
31564,REC-0031565,LOT-0000158,2020/21,2020,12,Kenya,Smallholder,1.23,19.0,1237.0,...,37821.0,Light,9.72,7.79,8.79,8.43,8.46,83.54,Excellent,4.52
19958,REC-0019959,LOT-0000158,2020/21,2020,12,Kenya,Smallholder,1.23,19.0,1237.0,...,37821.0,Light,9.72,7.79,8.79,8.43,8.46,83.54,Excellent,4.52
21601,REC-0021602,LOT-0000237,2021/22,2021,12,Vietnam,Estate,32.44,13.0,797.0,...,3600.0,Medium,7.25,7.82,7.30,7.21,7.47,69.96,Commercial,4.78
24039,REC-0024040,LOT-0000237,2021/22,2021,12,Vietnam,Estate,32.44,13.0,797.0,...,3600.0,Medium,7.25,7.82,7.30,7.21,7.47,69.96,Commercial,4.78


In [135]:
df[df["lot_id"] == "LOT-0000158"].T

,19958,31564
record_id,REC-0019959,REC-0031565
lot_id,LOT-0000158,LOT-0000158
harvest_season,2020/21,2020/21
harvest_calendar_year,2020,2020
harvest_month,12,12
country_of_origin,Kenya,Kenya
farm_type,Smallholder,Smallholder
farm_size_hectares,1.23,1.23
tree_age_years,19.0,19.0
altitude_mean_meters,1237.0,1237.0


In [136]:
meta = pd.read_csv("coffee_record_metadata.csv")
originals = meta[meta["record_type"] == "Original"]["record_id"]
df_clean = df[df["record_id"].isin(originals)].copy()
print(df_clean.shape)
print(df_clean["lot_id"].is_unique)

(35000, 36)
True


**Ризик 2: дублікати партій.** `df.duplicated()` показує 0, бо в кожного рядка свій `record_id`, але `lot_id` повторюється 700 разів.

**Як виявили:** `df["lot_id"].duplicated().sum()` → 700; `df.drop(columns=["record_id"]).duplicated().sum()` → 350 точних копій; порівняння партій через `.T` → у решти 350 відрізняються лише `moisture_pct` і `defect_rate_pct` (повторне вимірювання).

**Рішення:** лишаю лише записи з типом `Original` (за `coffee_record_metadata.csv`): 35 000 рядків, кожна партія один раз. Точні копії подвоювали б партії у статистиці. Для ревізій невідомо, яке вимірювання точніше, тому беру первинне й записую це як обмеження.

### Пропуски після очищення

In [137]:
missing = df_clean.isna().sum()
missing_pct = (df_clean.isna().mean() * 100).round(1)
pd.DataFrame({"missing": missing, "pct": missing_pct})[missing > 0]

,missing,pct
avg_annual_rainfall_mm,1777,5.1
humidity_pct,2239,6.4
moisture_pct,1587,4.5
bean_density_g_ml,1607,4.6
certification,21296,60.8


### Ризик 3: «пропуски» в `certification`

In [64]:
df_clean["certification"].value_counts(dropna=False)

certification
NaN                    21296
Fair Trade              5786
Organic                 3566
Multiple                2751
Rainforest Alliance     1601
Name: count, dtype: int64

In [91]:
raw = pd.read_csv("coffee_value_benchmark.csv", keep_default_na=False)
raw_clean = raw[raw["record_id"].isin(originals)]
raw_clean["certification"].value_counts()

certification
None                   21296
Fair Trade              5786
Organic                 3566
Multiple                2751
Rainforest Alliance     1601
Name: count, dtype: int64

In [118]:
df_clean["certification"] = df_clean["certification"].fillna("No certification")
df_clean["certification"].value_counts()

certification
No certification       21296
Fair Trade              5786
Organic                 3566
Multiple                2751
Rainforest Alliance     1601
Name: count, dtype: int64

**Ризик 3: «пропуски» в `certification`.**

**Як виявили:** `isna().sum()` показав 21 296 пропусків (60.8 %). Відкрила CSV у Блокноті – там записано слово `None`. Порівняла `value_counts(dropna=False)` з читанням через `keep_default_na=False`: кількість `NaN` дорівнює кількості `None`.

**Результат:** Pandas за замовчуванням читає слово `None` як пропуск, хоча автор мав на увазі «без сертифікату».

**Рішення:** заповнюю `fillna("No certification")`, бо це справжнє значення, а не невідомість; інакше аналіз сертифікації втратив би 61 % партій.

### Ризик 4: пропуски в числових стовпцях
Що змінює заповнення медіаною (на прикладі `humidity_pct`):

In [138]:
hum = df_clean["humidity_pct"]
print(hum.count(), hum.mean(), hum.median())
filled = hum.fillna(hum.median())
print(filled.count(), filled.mean(), filled.median())

32761 74.53998046457677 74.3
35000 74.52462857142856 74.3


**Ризик 4: пропуски в числових стовпцях** (`avg_annual_rainfall_mm`, `humidity_pct`, `moisture_pct`, `bean_density_g_ml`: 4.5–6.4 %).

**Як виявили:** `isna().sum()` і частка `(isna().mean() * 100).round(1)`; порівняння до й після `fillna(median)`.

**Результат:** після заповнення медіаною середнє `humidity_pct` змінилося лише на 0.015, медіана не змінилася, а `count` зріс на 2 239.

**Рішення:** лишаю `NaN` і записую, скільки їх. Заповнення не змінює статистику, а лише створює штучну впевненість: таблиця «вдає», що знає вологість для всіх партій. `mean()` і `median()` пропуски й так пропускають.

### Перевірка категорій

In [139]:
df_clean["country_of_origin"].value_counts()

country_of_origin
Brazil       11269
Vietnam       6348
Colombia      5264
Ethiopia      3507
Indonesia     3452
Honduras      2761
Kenya         2399
Name: count, dtype: int64

Кожна країна записана одним способом (7 значень), різних написань немає – дій не потрібно.

### Ризик 5: викиди в ціні

In [140]:
df_clean["price_usd_per_kg"].describe()

count    35000.000000
mean         4.811834
std          2.195118
min          1.780000
25%          3.460000
50%          4.190000
75%          5.360000
max         22.850000
Name: price_usd_per_kg, dtype: float64

In [141]:
expensive = df_clean[df_clean["price_usd_per_kg"] > 15]
print(len(expensive))
print(expensive["intended_buyer_segment"].value_counts())
print(expensive["traceability_level"].value_counts())
print(expensive["coffee_quality_score"].median(), df_clean["coffee_quality_score"].median())

173
intended_buyer_segment
Specialty    173
Name: count, dtype: int64
traceability_level
Microlot       154
Cooperative     10
Farm             9
Name: count, dtype: int64
81.51 78.23


**Ризик 5: викиди в `price_usd_per_kg`** (max 22.85 $/кг при медіані 4.19).

**Результат:**
- лише 173 партії із 35 000 (0.5 %) дорожчі за 15 $/кг;
- усі 173 призначені для ринку спешелті-кави (Specialty), де платять найбільше;
- 154 з них (89 %) – мікролоти, найдорожчий рівень простежуваності;
- їхня медіана якості 81.51 проти 78.23 загалом.

**Рішення:** не видаляю, бо це реальна преміальна кава, а не помилка введення. Для «типової ціни» використовую медіану, бо середнє ці партії тягнуть угору.

### Підсумок ризиків
1. Викиди у `farm_size_hectares` (плантації)
2. Дублікати партій за `lot_id` (700)
3. `None` у `certification` читається як пропуск
4. Пропуски в числових стовпцях, а заповнення дає штучну впевненість
5. Викиди в `price_usd_per_kg` (спешелті)

Повна таблиця: `lesson3_checklist_DYACHENKO.md`.

## Статистика двох стовпців (на `df_clean`)

In [142]:
price = df_clean["price_usd_per_kg"]
print("count:", price.count())
print("min:", price.min(), "| max:", price.max(), "| range:", round(price.max() - price.min(), 2))
print("mean:", round(price.mean(), 2), "| median:", price.median())
print("mode:", price.mode().tolist())
print("std:", round(price.std(), 2))

count: 35000
min: 1.78 | max: 22.85 | range: 21.07
mean: 4.81 | median: 4.19
mode: [3.61]
std: 2.2


In [143]:
quality = df_clean["coffee_quality_score"]
print("count:", quality.count())
print("min:", quality.min(), "| max:", quality.max(), "| range:", round(quality.max() - quality.min(),2))
print("mean:", round(quality.mean(), 2), "| median:", quality.median())
print("mode:", quality.mode().tolist())
print("std:", round(quality.std(), 2))

count: 35000
min: 60.0 | max: 94.18 | range: 34.18
mean: 77.6 | median: 78.23
mode: [78.3]
std: 4.92


Типова оцінка якості – близько 78 балів (медіана 78.23, мода 78.3). Середнє трохи нижче (77.6), бо кілька слабких партій тягнуть його вниз, тобто перекіс протилежний до ціни. Розкид помірний (std 4.92): більшість партій лежить приблизно між 73 і 83 балами при діапазоні 60–94.

In [144]:
r = df_clean["bean_density_g_ml"].corr(df_clean["coffee_quality_score"])
print(round(r, 2))

0.89


Щільність зерна і оцінка якості сильно пов'язані (r = 0.89): щільніші зерна зазвичай отримують вищу оцінку. Але кореляція не доводить причину: і щільність, і якість можуть залежати від третього чинника, наприклад висоти ферми.

## C. Приватність
Реєстр усіх 36 стовпців – у `lesson3_inventory_DYACHENKO.md`. Імен у наборі немає, але країна + тип ферми + площа + висота разом можуть вказати на конкретне господарство, а ціна й обсяг партії комерційно чутливі. Тому для звітів прибираю технічні ідентифікатори (`record_id`, `lot_id`), площу й висоту показую діапазонами, а результати публікую лише агреговано для груп з достатньою кількістю ферм.

## Нотатка про налагодження
**Помилка:** `TypeError: 'DataFrame' object is not callable` у рядку `df("price_usd_per_kg").count()`.

**Причина:** круглі дужки після імені означають «виконай», тож Python спробував викликати таблицю як функцію.

**Виправлення:** стовпець вибирають квадратними дужками: `df_clean["price_usd_per_kg"].count()`.

**Висновок:** `()` – щось робиться, `[]` – щось вибирається.

## Людські рішення та AI
Найважливіші рішення в цій роботі ухвалювала я: вважати `None` у `certification` значенням «без сертифікату», а не пропуском; лишити лише оригінальні записи партій і відкинути ревізії вимірювань; не видаляти великі ферми й дорогі партії, бо перевірка показала, що це реальні плантації та спешелті-кава; лишити пропуски в числових стовпцях як `NaN`, бо заповнення медіаною не змінює статистику, а лише створює штучну впевненість. AI я використовувала для пояснень і підказок (Claude) та автодоповнення коду (Copilot); кожну підказку перевіряла запуском коду або відкривши сам CSV. Подробиці – в `ai_journal.txt`.